In [ ]:
!pip install -q bitsandbytes accelerate

In [ ]:

bangla-med-qa.csv
bangla-med-qa.csv
CSV

#!/usr/bin/env python3 import os import time import torch import pandas as pd from tqdm import tqdm from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # ── Config ──────────────────────

pasted

i have given you the dataset and the one shot code, i have benchmarked this dataset using this one shot code, now i want to benchmark the same dataset using few shot, what should i do?

Ran 2 commands
Good — I can see the data shape now (994 rows, single-letter answers mostly, Bangla text, exam_name field for exam/year). Here's what to change to go from one-shot → few-shot.

The core idea
Right now your prompt is:

[SYSTEM]
Question: ...
A) ...
Answer:
Few-shot means prepending k solved examples (question + correct answer) before the actual test question, so the model sees the pattern demonstrated rather than just described. Two design decisions matter most:

Where do the few-shot examples come from? They must never be the row you're currently testing (that's leakage). Best practice: carve out a small fixed "exemplar pool" (e.g. 20–50 rows) before the benchmark loop, exclude that pool from the evaluation set, and sample k examples from it for every test row (or use the same fixed k examples for all rows — simpler and more reproducible).
Fixed exemplars vs. random-per-row exemplars? Fixed is far more common and much easier to reason about/debug — I'd recommend that unless you specifically want to study variance from example selection.
Concrete changes to your script
a) Build an exemplar pool and remove it from the eval set:

python
NUM_SHOTS = 4  # pick k

def build_few_shot_pool(df, k, seed=42):
    pool = df.sample(n=k, random_state=seed)
    remaining = df.drop(pool.index).reset_index(drop=True)
    return pool.reset_index(drop=True), remaining
b) Format each exemplar the same way as the real question, but with the answer filled in:

python
def build_exemplar_block(row):
    q = USER_TEMPLATE.format(
        question=row["question"], opt_a=row["options/A"], opt_b=row["options/B"],
        opt_c=row["options/C"], opt_d=row["options/D"],
    )
    return f"{q} {row['answer']}"
c) Assemble the few-shot prompt. Two ways to inject it — either as extra turns in the chat template (cleaner, lets the model's chat formatting handle role boundaries) or all folded into one user message. I'd use alternating user/assistant turns so it looks like real dialogue:

python
def _build_prompt(tokenizer, exemplar_rows, user_message):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in exemplar_rows:
        ex_q = USER_TEMPLATE.format(
            question=ex["question"], opt_a=ex["options/A"], opt_b=ex["options/B"],
            opt_c=ex["options/C"], opt_d=ex["options/D"],
        )
        messages.append({"role": "user", "content": ex_q})
        messages.append({"role": "assistant", "content": ex["answer"]})
    messages.append({"role": "user", "content": user_message})
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        # fallback: flatten into single user turn if template rejects multi-turn/system
        flat = SYSTEM_PROMPT + "\n\n"
        for i in range(1, len(messages)-1, 2):
            flat += messages[i]["content"] + " " + messages[i+1]["content"] + "\n\n"
        flat += messages[-1]["content"]
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": flat}], tokenize=False, add_generation_prompt=True
        )
d) Thread exemplar_rows through call_local_model → run_model_on_dataset → main, and pick NUM_SHOTS exemplars once at the top of main():

python
exemplar_pool, df = build_few_shot_pool(df, NUM_SHOTS)
exemplar_rows = exemplar_pool.to_dict("records")
then pass exemplar_rows into every call_local_model(...) call.

e) Keep a separate checkpoint/output path so few-shot results don't overwrite your one-shot ones:

python
OUTPUT_DIR = "/kaggle/working/bangla_med_benchmark_fewshot"
Practical notes
Context length: TigerLLM-9B likely has a few-thousand-token context; with Bangla text (which tokenizes less efficiently than English) 4–8 shots is a reasonable starting range. Test with NUM_SHOTS=4 first and check you're not truncating.
Balance the exemplars across A/B/C/D so you're not accidentally biasing the model toward one letter (worth doing e.g. stratified sampling: one example per correct letter, or 2 per letter for 8-shot).
Exclude multi-answer rows from the exemplar pool unless you deliberately want to demonstrate the comma-separated format — otherwise the model may default to single-letter answers even for the few multi-answer test questions.
Run one-shot and few-shot as separate scripts/outputs, then diff benchmark_results.csv from each to compare.
Want me to write out the complete modified script as a file so you can just swap it in for your Kaggle run?

give me the final code to do the task, no explanations

Created a file, shared files
Update CSV_PATH to your Kaggle input path and adjust NUM_SHOTS as needed.


Benchmark fewshot
PY 
again this error repeats for infinite timesmd-nishat-008/TigerLLM-9B-it:   0%|          | 0/986 [00:00<?, ?row/s]MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization

Diagnosing a repeating warning during model generation calls.
Diagnosing a repeating warning during model generation calls.
That message isn't an infinite loop — it's a warning printed once per generate() call, and your retry logic calls generate() up to 4 times per row when the model returns an unparseable/empty answer. With 986 rows that logs a lot. Two separate things to fix:

1. Silence the noisy bitsandbytes warning (harmless, just cast-dtype notice):

python
import warnings, logging
warnings.filterwarnings("ignore", message="MatMul8bitLt: inputs will be cast")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
Put this near the top of the script, after the imports.

2. Fix the actual dtype mismatch causing it — you're loading the model in bfloat16 but 8-bit quant internally computes in float16, so every matmul triggers a cast + warning. Match them:

python
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
)
...
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
(bfloat16 was added in your one-shot script specifically to avoid garbage output on Gemma-based models — TigerLLM may or may not need that; if outputs looked fine in fp16 originally, switching back removes the cast entirely.)

3. The real problem is likely the repeated retries themselves — if pred=None for many rows, that means the model is outputting empty/garbage answers with few-shot prompting, which is what's flooding your logs. That's worth checking directly:

Ran a command
Add a quick diagnostic print right after decoding, before parsing, so you can see what's actually coming back:

python
content = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
if not content:
    print(f"[DEBUG] empty generation — input token len={inputs.input_ids.shape[-1]}")
Likely cause with few-shot: your prompt is now much longer (system + k exemplars + question), and if it's pushing close to/over the model's max context length, generation can silently truncate or produce empty output. Check inputs.input_ids.shape[-1] — if it's large (e.g. >3000–4000 tokens depending on TigerLLM's context window), reduce NUM_SHOTS (try 2 instead of 4) and re-test on a handful of rows before running the full 986.

make proper changes to the code such that this dissapears, previously in some other benhcmarking i used warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

Edited a file benchmark_fewshot.py
Done — added right after the imports so it takes effect before the model/quantization code runs.

md-nishat-008/TigerLLM-9B-it:   9%|▉         | 87/986 [11:43<2:01:55,  8.14s/row]
[86] raw='A,C' → pred=AC | label=A | multi=False | correct=False, what do you make of this prediction?
Diagnosing why the model gave multiple letters for one answer.
Diagnosing why the model gave multiple letters for one answer.
That's not a bug in your code — it's the model genuinely answering wrong. Here's the breakdown:

raw='A,C' — the model actually generated two letters separated by a comma.
normalize_label correctly parsed that into 'AC' (sorted, deduped).
The true answer for that question is just 'A' (single-answer).
Since 'AC' != 'A', it's marked correct=False. That's expected/correct behavior from your scoring logic.
The interesting question is why the model produced two letters for a single-answer question. A few possibilities, roughly in order of likelihood:

Genuine confusion on the content — it's unsure between A and C and hedges by outputting both, since your system prompt explicitly allows multi-letter format ("If there are multiple correct answers, output the letters...").
Few-shot leakage effect — if your exemplar pool happened to have skewed formatting or the model is pattern-matching surface structure loosely, it might default to "give more than one letter" more often than the base model did in one-shot. This is worth checking.
Not a parsing artifact — the raw text really is "A,C", so this isn't a bug in parse_response.
To figure out if this is a one-off or systematic, check how often the model outputs multi-letter predictions for questions whose true label is single-answer:

Ran a command
Add this after the run finishes (or run it on predictions.csv once done) to quantify it:

python
df = pd.read_csv(os.path.join(OUTPUT_DIR, "predictions.csv"))
false_multi = df[(df["is_multi"] == False) & (df["prediction"].astype(str).str.len() > 1)]
print(f"Single-answer questions where model gave >1 letter: {len(false_multi)} / {(~df['is_multi']).sum()}")
print(false_multi[["question", "label", "prediction", "raw_response"]].head(10))
If this is happening on a meaningful fraction of single-answer rows (say >5–10%), it suggests the few-shot exemplars or system prompt are nudging the model toward over-hedging, and you could try:

Including only single-answer exemplars in the few-shot pool (which your current build_few_shot_pool already does — good), or
Tightening the system prompt, e.g. adding "Most questions have exactly one correct answer — only give multiple letters if you are certain the question requires it."
But if it's rare (a handful of rows out of 986), it's just normal model error and not worth chasing.

isnt my system a bit slow taking 8.5s per row?

Setting up a quantized model run with retry limits.
Setting up a quantized model run with retry limits.
8.14s/row is on the slower side for a 9B model doing single-token-ish generation — a few things are likely compounding:

Likely culprits, roughly by impact:

8-bit quantization (bitsandbytes) is compute-slow, not just memory-light. int8 matmul with dequant overhead is often slower per-token than plain fp16/bf16 inference, especially during prefill. People quantize for memory savings, not speed — if your GPU has enough VRAM (9B model is ~18GB in fp16), skipping quantization entirely can be significantly faster.
Few-shot made your prompts much longer, and prefill cost scales with prompt length. 4 exemplars of Bangla text (which tokenizes less efficiently than English/Latin scripts) could easily add 1000+ tokens to the prefill for every single row.
max_new_tokens=20 is generous for a 1–2 character answer — this alone doesn't matter much for latency if the model stops early via EOS, but if it's not emitting EOS cleanly it may generate the full 20 tokens every time.
Retries multiply cost — any row where pred is None triggers a full second (or third/fourth) generate call, and each one repeats the same expensive prefill.
Concrete things to try, cheapest first:

Reduce NUM_SHOTS (e.g. 4 → 2) — directly cuts prefill length.
Drop max_new_tokens to ~8 — plenty for "A", "A,B", etc.
Switch off 8-bit and load in bf16/fp16 directly if your GPU has ≥20GB VRAM:
python
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
Check what GPU you're actually on — this matters a lot:
Ran a command
Run !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv in your Kaggle notebook — if you're on a T4 (common free-tier Kaggle GPU), that alone explains a lot of the slowness regardless of quantization choice; T4s are notably slow for LLM inference compared to A100/L4/P100. If you're on T4 with only ~16GB, you may be stuck with 8-bit for memory reasons, in which case the biggest wins are cutting NUM_SHOTS and max_new_tokens.

Want me to update the script with max_new_tokens=8 and make NUM_SHOTS/quantization easily toggleable so you can A/B the speed?



You are out of free messages until 6:20 AM
Write a message…


Claude is AI and can make mistakes. Please double-check responses.
Benchmark fewshot · PY
#!/usr/bin/env python3
import os
import time
import warnings
import logging
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
 
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
 
# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = "/kaggle/input/datasets/syedmdnafissameen/medqa-new/bangla-med-qa.csv"
MODEL            = "md-nishat-008/TigerLLM-9B-it"
OUTPUT_DIR       = "/kaggle/working/bangla_med_benchmark_fewshot"
NUM_SAMPLES      = None
CHECKPOINT_EVERY = 20
NUM_SHOTS        = 4
FEWSHOT_SEED     = 42
 
VALID_LETTERS = ["A", "B", "C", "D"]
 
SYSTEM_PROMPT = """You are answering a multiple-choice question. Output only the correct option letter(s). If there is one correct answer, output exactly one of A, B, C, or D. If there are multiple correct answers, output the letters in alphabetical order separated by commas (e.g., A,B or B,D or A,B,C). Output nothing else. Never give blank output."""
 
USER_TEMPLATE = """Question: {question}
A) {opt_a}
B) {opt_b}
C) {opt_c}
D) {opt_d}
Answer:"""
 
 
# ── Label helpers ─────────────────────────────────────────────────────────────
def normalize_label(raw):
    if raw is None:
        return None
    found = sorted(set(ch for ch in str(raw).upper() if ch in VALID_LETTERS))
    return "".join(found) if found else None
 
 
def parse_response(raw_text):
    return normalize_label(raw_text)
 
 
def build_user_message(row):
    return USER_TEMPLATE.format(
        question=row["question"],
        opt_a=row["options/A"],
        opt_b=row["options/B"],
        opt_c=row["options/C"],
        opt_d=row["options/D"],
    )
 
 
# ── Few-shot exemplar pool ────────────────────────────────────────────────────
def build_few_shot_pool(df, k, seed=FEWSHOT_SEED):
    single_ans = df[df["answer"].astype(str).str.len() == 1]
    pool_parts = []
    per_letter = max(1, k // len(VALID_LETTERS))
    for letter in VALID_LETTERS:
        subset = single_ans[single_ans["answer"] == letter]
        n = min(per_letter, len(subset))
        if n > 0:
            pool_parts.append(subset.sample(n=n, random_state=seed))
    pool = pd.concat(pool_parts).sample(frac=1, random_state=seed) if pool_parts else single_ans.sample(n=0)
    if len(pool) < k:
        remaining_pool_candidates = single_ans.drop(pool.index)
        extra_n = min(k - len(pool), len(remaining_pool_candidates))
        if extra_n > 0:
            extra = remaining_pool_candidates.sample(n=extra_n, random_state=seed)
            pool = pd.concat([pool, extra])
    pool = pool.head(k).reset_index(drop=True)
    remaining = df.drop(df.index.intersection(df.merge(pool[["question"]], on="question", how="inner").index))
    remaining = df[~df["question"].isin(pool["question"])].reset_index(drop=True)
    return pool.reset_index(drop=True), remaining
 
 
def build_exemplar_messages(exemplar_rows):
    messages = []
    for ex in exemplar_rows:
        q = USER_TEMPLATE.format(
            question=ex["question"], opt_a=ex["options/A"], opt_b=ex["options/B"],
            opt_c=ex["options/C"], opt_d=ex["options/D"],
        )
        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": ex["answer"]})
    return messages
 
 
# ── Local HF Model Initialization & Call ──────────────────────────────────────
def load_local_model(model_name):
    print("Loading 8-bit quantized model...")
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16,
    )
 
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer
 
 
def _build_prompt(tokenizer, exemplar_rows, user_message):
    exemplar_messages = build_exemplar_messages(exemplar_rows)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + exemplar_messages + \
               [{"role": "user", "content": user_message}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        flat = SYSTEM_PROMPT + "\n\n"
        for i in range(0, len(exemplar_messages), 2):
            flat += exemplar_messages[i]["content"] + " " + exemplar_messages[i + 1]["content"] + "\n\n"
        flat += user_message
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": flat}], tokenize=False, add_generation_prompt=True
        )
 
 
def call_local_model(model, tokenizer, exemplar_rows, user_message):
    start = time.monotonic()
 
    try:
        prompt = _build_prompt(tokenizer, exemplar_rows, user_message)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
 
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
 
        generated_tokens = output_ids[0][inputs.input_ids.shape[-1]:]
        content = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        latency = time.monotonic() - start
 
        return content, None, latency
 
    except Exception as e:
        latency = time.monotonic() - start
        return None, f"execution_error:{e}", latency
 
 
# ── Checkpoint ────────────────────────────────────────────────────────────────
def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")
 
def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None
 
def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)
 
 
# ── Main loop ─────────────────────────────────────────────────────────────────
def run_model_on_dataset(model_obj, tokenizer, model_name, df, exemplar_rows, out_dir):
    existing  = load_checkpoint(out_dir, model_name)
    rows      = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows)
 
    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows
 
    pbar = tqdm(range(start_idx, len(df)), desc=model_name, unit="row",
                initial=start_idx, total=len(df))
 
    for i in pbar:
        row      = df.iloc[i]
        label    = normalize_label(row["answer"])
        user_msg = build_user_message(row)
 
        raw, err, latency = call_local_model(model_obj, tokenizer, exemplar_rows, user_msg)
        pred = parse_response(raw)
 
        retries = 0
        while pred is None and retries < 3:
            print(f"[{i}] pred=None (raw='{raw}', err='{err}') — retrying…")
            raw, err, latency = call_local_model(model_obj, tokenizer, exemplar_rows, user_msg)
            pred = parse_response(raw)
            retries += 1
 
        is_multi  = len(label) > 1 if label else False
        correct   = (pred == label) if pred is not None else False
 
        print(f"[{i}] raw='{raw}' → pred={pred} | label={label} | multi={is_multi} | correct={correct}")
 
        rows.append({
            "serial_no":    row.get("serial_no", i),
            "exam_name":    row.get("exam_name", ""),
            "question":     row["question"],
            "label":        label,
            "prediction":   pred,
            "is_multi":     is_multi,
            "correct":      correct,
            "latency":      latency,
            "raw_response": raw,
            "error":        err,
        })
 
        if (i - start_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(out_dir, model_name, rows)
 
    save_checkpoint(out_dir, model_name, rows)
    return rows
 
 
# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(rows):
    def _metrics(subset):
        valid = [r for r in subset if r["prediction"] is not None and str(r["prediction"]).strip() not in ("", "nan")]
        if not valid:
            return {"accuracy": None, "precision": None, "recall": None, "f1": None, "valid": 0, "total": len(subset)}
        yt = [str(r["label"])      for r in valid]
        yp = [str(r["prediction"]) for r in valid]
        classes = sorted(set(yt) | set(yp))
        return {
            "accuracy":  accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, labels=classes, average="macro", zero_division=0),
            "recall":    recall_score(yt, yp,    labels=classes, average="macro", zero_division=0),
            "f1":        f1_score(yt, yp,        labels=classes, average="macro", zero_division=0),
            "valid":     len(valid),
            "total":     len(subset),
        }
 
    latencies    = [r["latency"] for r in rows if r["latency"] is not None]
    single_rows  = [r for r in rows if not r["is_multi"]]
    multi_rows   = [r for r in rows if r["is_multi"]]
 
    return {
        "overall":       _metrics(rows),
        "single_answer": _metrics(single_rows),
        "multi_answer":  _metrics(multi_rows),
        "avg_latency":   sum(latencies) / len(latencies) if latencies else None,
    }
 
 
# ── Entry point ───────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    df = pd.read_csv(CSV_PATH)
    required = {"question", "options/A", "options/B", "options/C", "options/D", "answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required}")
 
    df["answer"] = df["answer"].apply(normalize_label)
 
    exemplar_pool, df = build_few_shot_pool(df, NUM_SHOTS)
    exemplar_rows = exemplar_pool.to_dict("records")
    print(f"Few-shot exemplars ({len(exemplar_rows)}):")
    for ex in exemplar_rows:
        print(f"  [{ex['answer']}] {ex['question'][:60]}")
 
    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)
 
    print(f"Dataset (excluding exemplars): {len(df)} rows")
    print(f"Single-answer: {(df['answer'].str.len() == 1).sum()} | Multi-answer: {(df['answer'].str.len() > 1).sum()}")
 
    model_obj, tokenizer = load_local_model(MODEL)
 
    rows    = run_model_on_dataset(model_obj, tokenizer, MODEL, df, exemplar_rows, OUTPUT_DIR)
    metrics = compute_metrics(rows)
 
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")
 
    results_rows = []
    for split, m in metrics.items():
        if split == "avg_latency":
            continue
        results_rows.append({
            "model":       MODEL,
            "num_shots":   NUM_SHOTS,
            "split":       split,
            "accuracy":    m.get("accuracy"),
            "precision":   m.get("precision"),
            "recall":      m.get("recall"),
            "f1":          m.get("f1"),
            "valid":       m.get("valid"),
            "total":       m.get("total"),
            "avg_latency": metrics["avg_latency"],
        })
 
    results_df = pd.DataFrame(results_rows)
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")
 
    print("\n── Results ──")
    print(results_df.to_string(index=False))
 
 
if __name__ == "__main__":
    main()
 



